# UCS761 - Deep Learning Lab 1
### Perceptron as the Smallest Decision-Making Model

This lab is about building a perceptron completely from scratch and training it on logic gates: AND, OR, NAND, NOR.

The whole point is not really the gates. The point is to see that the same exact code with different data gives completely different behavior. That is what data-driven learning actually means.

No ML libraries here. Just numpy.

In [1]:
import numpy as np


## The Perceptron

A perceptron takes two inputs, multiplies each by a weight, adds a bias, and passes the result through a step function.

If the weighted sum is >= 0, output is 1. Otherwise 0. That is literally the whole model.

In [2]:
def step(z):
    if z >= 0:
        return 1
    return 0

def predict(x1, x2, w1, w2, b):
    z = w1*x1 + w2*x2 + b
    return step(z)


## Training

The update rule is pretty straightforward:

```
error = actual - predicted
w1 = w1 + lr * error * x1
w2 = w2 + lr * error * x2
b  = b  + lr * error
```

If the prediction was already correct, error is 0 so nothing changes. If it was wrong, the weights shift a bit in the right direction.

In [3]:
def train(data, lr=0.1, epochs=100):
    w1 = 0.0
    w2 = 0.0
    b  = 0.0

    for ep in range(epochs):
        errors = 0
        for x1, x2, y in data:
            out   = predict(x1, x2, w1, w2, b)
            err   = y - out
            w1   += lr * err * x1
            w2   += lr * err * x2
            b    += lr * err
            errors += abs(err)

        if errors == 0:
            print(f"Converged at epoch {ep+1}")
            break

    return w1, w2, b


In [4]:
def show_results(gate_name, data, w1, w2, b):
    print(f"--- {gate_name} ---")
    print(f"Weights: w1={w1:.3f}  w2={w2:.3f}  bias={b:.3f}")
    for x1, x2, y in data:
        pred = predict(x1, x2, w1, w2, b)
        mark = "correct" if pred == y else "WRONG"
        print(f"  ({int(x1)}, {int(x2)}) -> {pred}  [expected {int(y)}]  {mark}")
    print()


## Q1: AND Gate

Output is 1 only when both inputs are 1.

In [5]:
and_gate = [
    (0, 0, 0),
    (0, 1, 0),
    (1, 0, 0),
    (1, 1, 1)
]

w1, w2, b = train(and_gate)
show_results("AND", and_gate, w1, w2, b)


Converged at epoch 4
--- AND ---
Weights: w1=0.200  w2=0.100  bias=-0.200
  (0, 0) -> 0  [expected 0]  correct
  (0, 1) -> 0  [expected 0]  correct
  (1, 0) -> 0  [expected 0]  correct
  (1, 1) -> 1  [expected 1]  correct



## Q2: OR Gate

Output is 1 when at least one input is 1.

In [6]:
or_gate = [
    (0, 0, 0),
    (0, 1, 1),
    (1, 0, 1),
    (1, 1, 1)
]

w1, w2, b = train(or_gate)
show_results("OR", or_gate, w1, w2, b)


Converged at epoch 4
--- OR ---
Weights: w1=0.100  w2=0.100  bias=-0.100
  (0, 0) -> 0  [expected 0]  correct
  (0, 1) -> 1  [expected 1]  correct
  (1, 0) -> 1  [expected 1]  correct
  (1, 1) -> 1  [expected 1]  correct



## Q3: NAND Gate

Opposite of AND. Output is 0 only when both inputs are 1.

In [7]:
nand_gate = [
    (0, 0, 1),
    (0, 1, 1),
    (1, 0, 1),
    (1, 1, 0)
]

w1, w2, b = train(nand_gate)
show_results("NAND", nand_gate, w1, w2, b)


Converged at epoch 6
--- NAND ---
Weights: w1=-0.200  w2=-0.100  bias=0.200
  (0, 0) -> 1  [expected 1]  correct
  (0, 1) -> 1  [expected 1]  correct
  (1, 0) -> 1  [expected 1]  correct
  (1, 1) -> 0  [expected 0]  correct



## Q4: NOR Gate

Opposite of OR. Output is 1 only when both inputs are 0.

In [8]:
nor_gate = [
    (0, 0, 1),
    (0, 1, 0),
    (1, 0, 0),
    (1, 1, 0)
]

w1, w2, b = train(nor_gate)
show_results("NOR", nor_gate, w1, w2, b)


Converged at epoch 4
--- NOR ---
Weights: w1=-0.100  w2=-0.100  bias=0.000
  (0, 0) -> 1  [expected 1]  correct
  (0, 1) -> 0  [expected 0]  correct
  (1, 0) -> 0  [expected 0]  correct
  (1, 1) -> 0  [expected 0]  correct



## Q5: XOR Gate: This One Fails

XOR is not linearly separable. No single straight line can separate the two output classes.
A single perceptron can only draw one decision boundary line. XOR needs two. This is exactly why multi-layer networks exist.

In [9]:
xor_gate = [
    (0, 0, 0),
    (0, 1, 1),
    (1, 0, 1),
    (1, 1, 0)
]

w1, w2, b = train(xor_gate, epochs=500)
show_results("XOR (will fail)", xor_gate, w1, w2, b)

print("XOR fails because it's not linearly separable.")
print("We'll need a hidden layer for this — covered in Lab 5.")


--- XOR (will fail) ---
Weights: w1=-0.100  w2=0.000  bias=0.000
  (0, 0) -> 1  [expected 0]  WRONG
  (0, 1) -> 1  [expected 1]  correct
  (1, 0) -> 0  [expected 1]  WRONG
  (1, 1) -> 0  [expected 0]  correct

XOR fails because it's not linearly separable.
We'll need a hidden layer for this — covered in Lab 5.


## Summary and Observations

**What worked:** AND, OR, NAND, NOR all converged correctly. Same 20-line training function, same learning rule, just the dataset changed.

**What did not:** XOR failed. Not a bug. It is a fundamental limit of single-layer perceptrons. The XOR data layout just cannot be split by one straight line.

**Effect of learning rate:**
Higher lr like 0.5 converges faster but can oscillate near the boundary. Lower lr like 0.01 is stable but needs many more epochs. 0.1 is a decent starting point for small datasets like this.

**Key observation:**
The architecture never changed at all. Same weights, same update rule, same everything. Only the data changed, and the model learned completely different decision boundaries each time. This is what 'data defines the task' actually means in practice.